# An LLM is one component of a decision system
This API-free exercise isolates two surrounding components: retrieving dated evidence and checking that a drafted claim cites the selected evidence.

## 1. Retrieval with scikit-learn
Use TF–IDF and cosine similarity to select a passage from a tiny reading room.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    '2026-02-10 filing: revenue rose 8 percent.',
    '2026-05-09 filing: operating margin fell to 14 percent.',
    '2026-05-12 call: management expects lower capital spending.'
]
query = 'What happened to operating margin?'
vectorizer = TfidfVectorizer().fit(documents + [query])
matrix = vectorizer.transform(documents + [query])
scores = cosine_similarity(matrix[-1], matrix[:-1]).ravel()
best = scores.argmax()
print('evidence:', documents[best])
print('required citation:', f'[document {best + 1}]')

## 2. Build retrieval and a citation check from scratch
Use token overlap to rank passages, then verify that the drafted answer names its evidence.

In [ ]:
import re

tokens = lambda text: set(re.findall(r'[a-z]+', text.lower()))
query_tokens = tokens(query)
overlap = [len(query_tokens & tokens(document)) for document in documents]
scratch_best = max(range(len(documents)), key=overlap.__getitem__)
answer = 'Operating margin fell to 14 percent. [document 2]'
required_citation = f'[document {scratch_best + 1}]'
print('evidence:', documents[scratch_best])
print('answer:', answer)
print('citation check:', required_citation in answer)

## What to notice
Retrieval and verification are separate from generation. A fluent answer without the right dated evidence should fail the system check.